# Evaluating Multiple LM Outputs (External)

In [1]:
%load_ext autoreload
%autoreload 2

In [18]:
# imports
import json
import os
import pandas as pd
import importlib.util
import sys
from os.path import join
from copy import deepcopy
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion
from stat_genie.blade_pipeline.additions.analysis.fix_code import check_and_fix_transform_model_code, check_and_fix_final_answer_code

In [4]:
# load files
analysis_subdir_path_1 = "analysis1_output"
analysis_subdir_path_2 = "analysis2_output"

multirun_filename_1 = "multirun_analyses.json"
multirun_filename_2 = "multirun_analyses.json"

# use both files to get analysis code paths
multirun_path_1 = join(analysis_subdir_path_1, multirun_filename_1)
multirun_path_2 = join(analysis_subdir_path_2, multirun_filename_2)

with open(multirun_path_1, "r") as file:
    multirun_analyses_1 = json.load(file)

with open(multirun_path_2, "r") as file:
    multirun_analyses_2 = json.load(file)

num_analyses_1 = multirun_analyses_1['n']
num_analyses_2 = multirun_analyses_2['n']

analysis_code_filenames_1 = [f"llm_analysis_{i}.py" for i in range(num_analyses_1)]
analysis_code_filenames_2 = [f"llm_analysis_{i}.py" for i in range(num_analyses_2)]

analysis_code_paths_1 = [join(analysis_subdir_path_1, filename)
                         for filename in analysis_code_filenames_1]

analysis_code_paths_2 = [join(analysis_subdir_path_2, filename)
                         for filename in analysis_code_filenames_2]

In [5]:
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-11-25 12:28:59.36][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [6]:
features_1 = format_features(multirun_analyses_1, num_analyses_1, llm_assistant)
features_2 = format_features(multirun_analyses_2, num_analyses_2, llm_assistant)

In [7]:
model_info_1 = format_model_info(multirun_analyses_1, num_analyses_1, llm_assistant)
model_info_2 = format_model_info(multirun_analyses_2, num_analyses_2, llm_assistant)

In [8]:
# load dataset, need more user-friendly input method later
dataset_name = multirun_analyses_1['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")
absolute_dataset_path = os.path.abspath(dataset_path)
data = pd.read_csv(dataset_path)

In [9]:
# check that code works
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    # get absolute path using relative path so that the helper function works correctly
    absolute_path = os.path.abspath(analysis_code_path)
    # call helper function to ensure code correctness
    num_iterations = check_and_fix_transform_model_code(f"llm_analysis_{i}",
                                        absolute_path,
                                        absolute_dataset_path,
                                        llm_provider,
                                        llm_model,
                                        verbose=False)
    print(f"Analysis 1 iteration {i} required {num_iterations} correction iterations.")
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    # get absolute path using relative path so that the helper function works correctly
    absolute_path = os.path.abspath(analysis_code_path)
    # call helper function to ensure code correctness
    num_iterations = check_and_fix_transform_model_code(f"llm_analysis_{i}",
                                        absolute_path,
                                        absolute_dataset_path,
                                        llm_provider,
                                        llm_model,
                                        verbose=False)
    print(f"Analysis 2 iteration {i} required {num_iterations} correction iterations.")

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


Analysis 1 iteration 0 required 0 correction iterations.


/accounts/grad/zachrewolinski/research/stat-genie/examples/feature_perturbation/analysis1_output/llm_analysis_1.py:162: UserWarning: No target column found among common names. Returning dataframe numeric summary instead of fitting a model.
  warnings.warn(


Analysis 1 iteration 1 required 0 correction iterations.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:        alldeaths_count   No. Observations:                   93
Model:                            GLM   Df Residuals:                       86
Model Family:        NegativeBinomial   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -352.18
Date:                Tue, 25 Nov 2025   Deviance:                       163.10
Time:                        12:29:27   Pearson chi2:                     255.
No. Iterations:                    14   Pseudo R-squ. (CS):             0.8405
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2288      0.028      8.278      0.0

In [10]:
transform_functions_1 = {}
transform_functions_2 = {}
model_functions_1 = {}
model_functions_2 = {}

# ----- get the transform and model functions for the first input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_1_{i}"] = module
    spec.loader.exec_module(module)
    
    transform_functions_1[i] = module.transform
    model_functions_1[i] = module.model

# ----- get the transform and model functions for the second input group -----
for i, analysis_code_path in enumerate(analysis_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_2_{i}"] = module
    spec.loader.exec_module(module)

    transform_functions_2[i] = module.transform
    model_functions_2[i] = module.model


In [11]:
transformed_datasets_1 = {}
for i, transform_func in transform_functions_1.items():
    try:
        transformed_datasets_1[i] = transform_func(data.copy())
        print(f"[Transform 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 1-{i}] Failed with error: {e}")
        transformed_datasets_1[i] = None

transformed_datasets_2 = {}
for i, transform_func in transform_functions_2.items():
    try:
        transformed_datasets_2[i] = transform_func(data.copy())
        print(f"[Transform 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform 2-{i}] Failed with error: {e}")
        transformed_datasets_2[i] = None


model_results_1 = {}
for i, model_func in model_functions_1.items():
    try:
        if transformed_datasets_1[i] is None:
            print(f"[Model 1-{i}] Skipping — transform failed.")
            continue

        model_results_1[i] = model_func(transformed_datasets_1[i].copy())
        print(f"[Model 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 1-{i}] Failed with error: {e}")
        model_results_1[i] = None

model_results_2 = {}
for i, model_func in model_functions_2.items():
    try:
        if transformed_datasets_2[i] is None:
            print(f"[Model 2-{i}] Skipping — transform failed.")
            continue

        model_results_2[i] = model_func(transformed_datasets_2[i].copy())
        print(f"[Model 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Model 2-{i}] Failed with error: {e}")
        model_results_2[i] = None


[Transform 1-0] Completed successfully.
[Transform 1-1] Completed successfully.
[Transform 1-2] Completed successfully.
[Transform 2-0] Completed successfully.
[Transform 2-1] Completed successfully.
[Transform 2-2] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 1-0] Completed successfully.
[Model 1-1] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/examples/feature_perturbation/analysis1_output/llm_analysis_1.py:162: UserWarning: No target column found among common names. Returning dataframe numeric summary instead of fitting a model.
  warnings.warn(
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


                 Generalized Linear Model Regression Results                  
Dep. Variable:        alldeaths_count   No. Observations:                   93
Model:                            GLM   Df Residuals:                       86
Model Family:        NegativeBinomial   Df Model:                            6
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -352.18
Date:                Tue, 25 Nov 2025   Deviance:                       163.10
Time:                        12:29:58   Pearson chi2:                     255.
No. Iterations:                    14   Pseudo R-squ. (CS):             0.8405
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2288      0.028      8.278      0.0

In [12]:
# # get absolute path of analysis_subdir_path_1
# absolute_analysis_subdir_path_1 = os.path.abspath(analysis_subdir_path_1)
# absolute_analysis_subdir_path_1

In [13]:
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)

task = info_json['research_questions']

for i in range(num_analyses_1):

    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    model_output = model_results_1[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_1),
        i,
        model_output
    )

for i in range(num_analyses_2):

    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']
    model_output = model_results_2[i]

    write_final_answer_code(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        os.path.abspath(analysis_subdir_path_2),
        i,
        model_output
    )


[2025-11-25 12:30:21.55][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:30:59.01][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  37.47 seconds
[2025-11-25 12:30:59.02][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-25 12:30:59.12][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:31:32.36][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  33.24 seconds
[2025-11-25 12:31:32.36][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-25 12:31:32.41][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:32:00.75][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  28.

In [14]:
answer_code_paths_1 = [join(analysis_subdir_path_1, f"llm_answer_{i}.py") for i in range(num_analyses_1)]
answer_code_paths_2 = [join(analysis_subdir_path_2, f"llm_answer_{i}.py") for i in range(num_analyses_2)]

In [15]:
# check that code works
for i, answer_code_path in enumerate(answer_code_paths_1):
    # get absolute path using relative path so that the helper function works correctly
    absolute_path = os.path.abspath(answer_code_path)
    # call helper function to ensure code correctness
    num_iterations = check_and_fix_final_answer_code(f"llm_answer_{i}",
                                        absolute_path,
                                        model_results_1[i],
                                        llm_provider,
                                        llm_model,
                                        verbose=True)
    print(f"Answer 1 iteration {i} required {num_iterations} correction iterations.")
for i, answer_code_path in enumerate(answer_code_paths_2):
    # get absolute path using relative path so that the helper function works correctly
    absolute_path = os.path.abspath(answer_code_path)
    # call helper function to ensure code correctness
    num_iterations = check_and_fix_final_answer_code(f"llm_answer_{i}",
                                        absolute_path,
                                        model_results_2[i],
                                        llm_provider,
                                        llm_model,
                                        verbose=True)
    print(f"Answer 2 iteration {i} required {num_iterations} correction iterations.")

Answer 1 iteration 0 required 0 correction iterations.
Answer 1 iteration 1 required 0 correction iterations.
Error during runtime:
Traceback (most recent call last):
  File "/accounts/grad/zachrewolinski/research/stat-genie/src/stat_genie/blade_pipeline/additions/analysis/fix_code.py", line 273, in is_final_answer_code_correct
    final_answer = final_answer_func(model_output)
  File "/accounts/grad/zachrewolinski/research/stat-genie/examples/feature_perturbation/analysis1_output/llm_answer_2.py", line 71, in extract_final_answer
    nobs = int(getattr(res, 'nobs', getattr(res, 'model', {}).get('nobs', None))) if hasattr(res, 'nobs') or hasattr(res, 'model') else None
AttributeError: 'GLM' object has no attribute 'get'

[2025-11-25 12:32:13.36][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-11-25 12:32:13.78][base.py:60 - stat_genie.blade_pipeline.llms.base:generate

In [16]:
# get final answer functions in dict
final_answer_functions_1 = {}
final_answer_functions_2 = {}

# ----- get the final answer functions for the first input group -----
for i, answer_code_path in enumerate(answer_code_paths_1):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_1_{i}"] = module
    spec.loader.exec_module(module)
    
    final_answer_functions_1[i] = module.extract_final_answer

# ----- get the final answer functions for the second input group -----
for i, answer_code_path in enumerate(answer_code_paths_2):
    spec = importlib.util.spec_from_file_location(f"llm_answer_{i}",
                                                  answer_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_answer_2_{i}"] = module
    spec.loader.exec_module(module)

    final_answer_functions_2[i] = module.extract_final_answer


In [19]:
final_answers_1 = {}
for i, final_answer_func in final_answer_functions_1.items():
    try:
        model_output = deepcopy(model_results_1[i])
        final_answers_1[i] = final_answer_func(model_output)
        print(f"[Answer 1-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 1-{i}] Failed with error: {e}")
        final_answers_1[i] = None

final_answers_2 = {}
for i, final_answer_func in final_answer_functions_2.items():
    try:
        model_output = deepcopy(model_results_2[i])
        final_answers_2[i] = final_answer_func(model_output)
        print(f"[Answer 2-{i}] Completed successfully.")
    except Exception as e:
        print(f"[Answer 2-{i}] Failed with error: {e}")
        final_answers_2[i] = None

[Answer 1-0] Completed successfully.
[Answer 1-1] Completed successfully.
[Answer 1-2] Completed successfully.
[Answer 2-0] Completed successfully.
[Answer 2-1] Completed successfully.
[Answer 2-2] Completed successfully.


In [21]:
code_filename = f"llm_answer_0.py"
code_path = os.path.abspath(os.path.join(analysis_subdir_path_1, code_filename))
with open(code_path, "r", encoding="utf-8") as f:
    interpretation_code_str = f.read()
interpretation_code_str

'def extract_final_answer(model_output):\n    """\n    Extract key statistics from the model output returned by the provided `model` function.\n\n    Returns a dict with:\n      - "object": a dict of extracted numeric results for each model found (nb_model, gender_model, damage_model)\n      - "description": a short plain-language interpretation about whether the evidence supports\n                       the hypothesis that more-feminine names are associated with different outcomes.\n\n    The function handles either:\n      - a dict with keys \'nb_model\', \'gender_model\', \'damage_model\' (as returned by the model function), or\n      - a single fitted model object (assumed to be the negative-binomial / primary model).\n    """\n    import numpy as np\n    import math\n\n    # Helper to safely extract a parameter-related summary from a fitted results object\n    def summarize_param(res, param_name):\n        out = {"present": False, "coef": None, "pvalue": None, "ci_lower": None, "c

In [22]:
conclusions_1 = {}

for i in range(num_analyses_1):
    independent_variable = features_1[i]['independent_variables']
    dependent_variable = features_1[i]['response_variables']

    model_code = multirun_analyses_1['analyses'][str(i)]['m_code']
    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_1
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_1, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_1[i]

    conclusions_1[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )


conclusions_2 = {}

for i in range(num_analyses_2):
    independent_variable = features_2[i]['independent_variables']
    dependent_variable = features_2[i]['response_variables']

    model_code = multirun_analyses_2['analyses'][str(i)]['m_code']

    # read in final answer code from file "llm_answer_{i}.py" at analysis_subdir_path_2
    code_filename = f"llm_answer_{i}.py"
    code_path = os.path.abspath(os.path.join(analysis_subdir_path_2, code_filename))
    with open(code_path, "r", encoding="utf-8") as f:
        interpretation_code_str = f.read()
    interpretation_output = final_answers_2[i] if i < len(final_answers_2) else None

    conclusions_2[i] = make_conclusion(
        llm_assistant,
        task,
        independent_variable,
        dependent_variable,
        model_code,
        interpretation_code_str,
        interpretation_output
    )


[2025-11-25 12:43:11.35][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:43:15.68][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.33 seconds
[2025-11-25 12:43:15.69][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-25 12:43:15.72][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:43:21.29][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.57 seconds
[2025-11-25 12:43:21.29][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-25 12:43:21.32][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:43:26.97][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.65 

In [24]:
llm_judge = llm(provider=llm_provider, model=llm_model)
data_head = data.head(10)

[2025-11-25 12:44:06.75][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [25]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final JSON object.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in JSON format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)


def make_judge_prompt(task, data_head, featA, featB, modelA, modelB, conclA, conclB):
    return (
        f"Research Question / Context:\n{task}\n\n"
        "Here is a sample of the dataset to understand the structure and variables:\n"
        f"{data_head}\n\n"
        "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
        "==================== TRIAL A ====================\n\n"
        "Independent Variables:\n"
        f"{featA['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featA.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featA['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelA}\n\n"
        "Conclusion:\n"
        f"{conclA}\n\n"
        "==================== TRIAL B ====================\n\n"
        "Independent Variables:\n"
        f"{featB['independent_variables']}\n\n"
        "Control Variables:\n"
        f"{featB.get('control_variables')}\n\n"
        "Response Variables:\n"
        f"{featB['response_variables']}\n\n"
        "Model Specification:\n"
        f"{modelB}\n\n"
        "Conclusion:\n"
        f"{conclB}\n\n"
        "Now, following your reasoning plan, provide similarity ratings as JSON only."
    )


In [26]:
judge_results = {}

num_comparisons = min(num_analyses_1, num_analyses_2)

for i in range(num_comparisons):

    user_prompt = make_judge_prompt(
        task, 
        data_head,
        features_1[i], features_2[i],
        model_info_1[i], model_info_2[i],
        conclusions_1[i], conclusions_2[i]
    )

    # call LLM judge
    result = llm_judge.generate([
        {"role": "system", "content": judge_system_prompt},
        {"role": "user", "content": user_prompt}
    ])

    judge_results[i] = result


[2025-11-25 12:44:18.68][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:44:25.50][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  6.83 seconds
[2025-11-25 12:44:25.51][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-25 12:44:25.57][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:44:33.19][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.62 seconds
[2025-11-25 12:44:33.19][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-25 12:44:33.29][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-25 12:44:42.17][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.88 

In [27]:
judge_results

{0: TextGenResponse(text=[Message(role='assistant', content='{\n  "independent_variables": 5,\n  "control_variables": 5,\n  "response_variables": 5,\n  "model_specification": 5,\n  "conclusions": 5,\n  "overall_similarity": 5\n}')], config=TextGenConfig(model='gpt-5-mini', n=1, temperature=0.8, max_tokens=None, top_p=None, top_k=None, run_config=None, stop_sequences=None, frequency_penalty=0.0, presence_penalty=0.0), api_elapsed_time=6.828932285308838, cache_elapsed_time=None, from_cache=False, response=ChatCompletion(id='chatcmpl-CfuBXJuZr5SXtOq2JAL6xcbVOTdl5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "independent_variables": 5,\n  "control_variables": 5,\n  "response_variables": 5,\n  "model_specification": 5,\n  "conclusions": 5,\n  "overall_similarity": 5\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764103459, model='gpt-5-mini-2025-08-07', object='c